In [ ]:
import pandas as pd

player_stats_df = pd.read_csv('/content/player_stats_aggregated.csv')
player_stats_defensive_df = pd.read_csv('/content/player_stats_defensive_aggregated.csv')
player_stats_offensive_df = pd.read_csv('/content/player_stats_offensive_aggregated.csv')
player_stats_passing_df = pd.read_csv('/content/player_stats_passing_aggregated.csv')

### Player Stats Aggregated

In [ ]:
display(player_stats_df.head())

,Player_ID,Player_Name,apps,manOfTheMatch,goal,assistTotal,yellowCard,redCard,shotsPerGame,aerialWonPerGame,passSuccess,rating
0,5583,Cristiano Ronaldo,38,0,33,0,2.0,1.0,6.200000,0.800000,91.397849,7.006000
1,8280,Craig Gordon,9,0,0,0,0.0,0.0,NaN,0.444444,52.843902,6.412778
2,11119,Lionel Messi,41,12,25,8,3.0,0.0,3.950000,0.171429,83.431002,7.812119
3,13754,Manuel Neuer,43,0,0,0,3.0,0.0,NaN,0.045455,79.393175,6.393606
4,20874,Luka Modric,55,4,4,5,4.0,0.0,0.876634,0.411765,89.435256,6.715343


### Player Stats Defensive Aggregated

In [ ]:
display(player_stats_defensive_df.head())

,Player_ID,Player_Name,apps,goalOwn,tacklePerGame,interceptionPerGame,foulsPerGame,offsideWonPerGame,clearancePerGame,wasDribbledPerGame,outfielderBlockPerGame
0,5583,Cristiano Ronaldo,38,0.0,NaN,NaN,0.600000,NaN,NaN,NaN,NaN
1,8280,Craig Gordon,8,0.0,NaN,NaN,NaN,NaN,1.055556,NaN,NaN
2,11119,Lionel Messi,41,0.0,0.946429,0.142857,0.435714,NaN,0.250000,1.059524,0.250000
3,13754,Manuel Neuer,43,0.0,NaN,NaN,NaN,NaN,0.863636,0.090909,NaN
4,20874,Luka Modric,55,0.0,0.736111,0.687908,0.421569,0.058824,0.698529,0.437092,0.238971


### Player Stats Offensive Aggregated

In [ ]:
display(player_stats_offensive_df.head())

,Player_ID,Team_Name,Player_Name,apps,goal,assistTotal,shotsPerGame,keyPassPerGame,dribbleWonPerGame,foulGivenPerGame,offsideGivenPerGame,dispossessedPerGame,turnoverPerGame,rating
0,5583,Portugal,Cristiano Ronaldo,38,33,0,6.200000,0.200000,NaN,NaN,1.000000,0.600000,0.600000,7.006000
1,8280,Scotland,Craig Gordon,8,0,0,NaN,NaN,NaN,0.333333,NaN,NaN,0.500000,6.412778
2,11119,Argentina,Lionel Messi,41,25,8,3.950000,2.516667,2.609524,1.604762,0.454762,2.223810,1.290476,7.812119
3,13754,Germany,Manuel Neuer,43,0,0,NaN,NaN,0.090909,0.393939,NaN,NaN,NaN,6.387705
4,20874,Croatia,Luka Modric,55,4,5,0.876634,1.097222,0.437500,0.545752,NaN,0.301471,0.613562,6.715343


### Player Stats Passing Aggregated

In [ ]:
display(player_stats_passing_df.head())

,Player_ID,Team_Name,Player_Name,apps,assistTotal,keyPassPerGame,totalPassesPerGame,accurateCrossesPerGame,accurateLongPassPerGame,accurateThroughBallPerGame,rating,passSuccess
0,5583,Portugal,Cristiano Ronaldo,38,0,0.200000,18.600000,0.200000,0.600000,NaN,7.006000,91.397849
1,8280,Scotland,Craig Gordon,8,0,NaN,27.666667,NaN,6.055556,NaN,6.412778,52.843902
2,11119,Argentina,Lionel Messi,41,8,2.516667,42.352381,1.092857,1.976190,0.642857,7.812119,83.431002
3,13754,Germany,Manuel Neuer,43,0,NaN,30.779545,NaN,6.434091,NaN,6.387705,81.135790
4,20874,Croatia,Luka Modric,55,5,1.097222,46.213235,0.687092,2.975490,0.264706,6.715343,89.435256


In [ ]:
# Create copies to avoid modifying original dataframes
df_general = player_stats_df.copy()
df_defensive = player_stats_defensive_df.copy()
df_offensive = player_stats_offensive_df.copy()
df_passing = player_stats_passing_df.copy()

# Defining the columns that should primarily come from df_general
general_stats_primary_cols = ['apps', 'goal', 'assistTotal', 'shotsPerGame', 'passSuccess', 'rating']

# Rename general stats columns in df_defensive, df_offensive, and df_passing to temporary names
# We will drop these temporary columns after merging to retain only df_general's versions
for df in [df_defensive, df_offensive, df_passing]:
    for col in general_stats_primary_cols:
        if col in df.columns:
            df.rename(columns={col: f'{col}_temp_general'}, inplace=True)

# Team_Name and keyPassPerGame should primarily come from df_offensive
# So, rename them in df_passing to temporary names
if 'Team_Name' in df_passing.columns:
    df_passing.rename(columns={'Team_Name': 'Team_Name_temp_passing'}, inplace=True)
if 'keyPassPerGame' in df_passing.columns:
    df_passing.rename(columns={'keyPassPerGame': 'keyPassPerGame_temp_passing'}, inplace=True)


# Merging Process

# Start with df_general (player_stats_df) as the base
final_merged_df = df_general.copy()

# Merge with df_defensive
# The temporary 'apps_temp_general' from df_defensive will be dropped
final_merged_df = pd.merge(final_merged_df, df_defensive, on=['Player_ID', 'Player_Name'], how='outer')
final_merged_df.drop(columns=[col for col in final_merged_df.columns if col.endswith('_temp_general') and col.startswith('apps')], errors='ignore', inplace=True)

# Merge with df_offensive
# Conflicting general stats are already renamed to '_temp_general' in df_offensive and will be dropped later
# 'Team_Name' and 'keyPassPerGame' from df_offensive should merge cleanly as they are not in df_general
final_merged_df = pd.merge(final_merged_df, df_offensive, on=['Player_ID', 'Player_Name'], how='outer')
# Drop all columns that ended with '_temp_general' from the offensive dataframe, as we prefer df_general's versions
cols_to_drop_offensive_temps = [f'{col}_temp_general' for col in general_stats_primary_cols]
final_merged_df.drop(columns=cols_to_drop_offensive_temps, errors='ignore', inplace=True)

# Merge with df_passing
# Conflicting general stats are already renamed to '_temp_general' in df_passing and will be dropped later
# 'Team_Name' and 'keyPassPerGame' from df_passing are renamed to '_temp_passing' and will be dropped
final_merged_df = pd.merge(final_merged_df, df_passing, on=['Player_ID', 'Player_Name'], how='outer')
# Drop general stats that were temporarily renamed in df_passing
cols_to_drop_passing_general_temps = [f'{col}_temp_general' for col in general_stats_primary_cols]
final_merged_df.drop(columns=cols_to_drop_passing_general_temps, errors='ignore', inplace=True)

# Drop 'Team_Name_temp_passing' and 'keyPassPerGame_temp_passing' to keep df_offensive's versions
final_merged_df.drop(columns=['Team_Name_temp_passing', 'keyPassPerGame_temp_passing'], errors='ignore', inplace=True)

# Assign the final result to merged_df
merged_df = final_merged_df

In [ ]:
pd.set_option('display.max_columns', None)
display(merged_df.head())
pd.reset_option('display.max_columns')

,Player_ID,Player_Name,apps,manOfTheMatch,goal,assistTotal,yellowCard,redCard,shotsPerGame,aerialWonPerGame,passSuccess,rating,goalOwn,tacklePerGame,interceptionPerGame,foulsPerGame,offsideWonPerGame,clearancePerGame,wasDribbledPerGame,outfielderBlockPerGame,Team_Name,keyPassPerGame,dribbleWonPerGame,foulGivenPerGame,offsideGivenPerGame,dispossessedPerGame,turnoverPerGame,totalPassesPerGame,accurateCrossesPerGame,accurateLongPassPerGame,accurateThroughBallPerGame
0,5583,Cristiano Ronaldo,38,0,33,0,2.0,1.0,6.200000,0.800000,91.397849,7.006000,0.0,NaN,NaN,0.600000,NaN,NaN,NaN,NaN,Portugal,0.200000,NaN,NaN,1.000000,0.600000,0.600000,18.600000,0.200000,0.600000,NaN
1,8280,Craig Gordon,9,0,0,0,0.0,0.0,NaN,0.444444,52.843902,6.412778,0.0,NaN,NaN,NaN,NaN,1.055556,NaN,NaN,Scotland,NaN,NaN,0.333333,NaN,NaN,0.500000,27.666667,NaN,6.055556,NaN
2,11119,Lionel Messi,41,12,25,8,3.0,0.0,3.950000,0.171429,83.431002,7.812119,0.0,0.946429,0.142857,0.435714,NaN,0.250000,1.059524,0.250000,Argentina,2.516667,2.609524,1.604762,0.454762,2.223810,1.290476,42.352381,1.092857,1.976190,0.642857
3,13754,Manuel Neuer,43,0,0,0,3.0,0.0,NaN,0.045455,79.393175,6.393606,0.0,NaN,NaN,NaN,NaN,0.863636,0.090909,NaN,Germany,NaN,0.090909,0.393939,NaN,NaN,NaN,30.779545,NaN,6.434091,NaN
4,20874,Luka Modric,55,4,4,5,4.0,0.0,0.876634,0.411765,89.435256,6.715343,0.0,0.736111,0.687908,0.421569,0.058824,0.698529,0.437092,0.238971,Croatia,1.097222,0.437500,0.545752,NaN,0.301471,0.613562,46.213235,0.687092,2.975490,0.264706


In [ ]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1208 entries, 0 to 1207
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Player_ID                   1208 non-null   int64  
 1   Player_Name                 1208 non-null   object 
 2   apps                        1208 non-null   int64  
 3   manOfTheMatch               1208 non-null   int64  
 4   goal                        1208 non-null   int64  
 5   assistTotal                 1208 non-null   int64  
 6   yellowCard                  1208 non-null   float64
 7   redCard                     1208 non-null   float64
 8   shotsPerGame                814 non-null    float64
 9   aerialWonPerGame            879 non-null    float64
 10  passSuccess                 924 non-null    float64
 11  rating                      924 non-null    float64
 12  goalOwn                     1157 non-null   float64
 13  tacklePerGame               852 n

In [ ]:
columns = merged_df.columns.tolist()

# Move 'Team_Name' to after 'Player_Name'
team_name_col = columns.pop(columns.index('Team_Name'))
player_name_index = columns.index('Player_Name')
columns.insert(player_name_index + 1, team_name_col)

# Move 'rating' to after 'Team_Name'
rating_col = columns.pop(columns.index('rating'))
team_name_index = columns.index('Team_Name')
columns.insert(team_name_index + 1, rating_col)

merged_df = merged_df[columns]

pd.set_option('display.max_columns', None)
display(merged_df.head())
pd.reset_option('display.max_columns')

print('\n DataFrame Info after column reorder')
merged_df.info()

,Player_ID,Player_Name,Team_Name,rating,apps,manOfTheMatch,goal,assistTotal,yellowCard,redCard,shotsPerGame,aerialWonPerGame,passSuccess,goalOwn,tacklePerGame,interceptionPerGame,foulsPerGame,offsideWonPerGame,clearancePerGame,wasDribbledPerGame,outfielderBlockPerGame,keyPassPerGame,dribbleWonPerGame,foulGivenPerGame,offsideGivenPerGame,dispossessedPerGame,turnoverPerGame,totalPassesPerGame,accurateCrossesPerGame,accurateLongPassPerGame,accurateThroughBallPerGame
0,5583,Cristiano Ronaldo,Portugal,7.006000,38,0,33,0,2.0,1.0,6.200000,0.800000,91.397849,0.0,NaN,NaN,0.600000,NaN,NaN,NaN,NaN,0.200000,NaN,NaN,1.000000,0.600000,0.600000,18.600000,0.200000,0.600000,NaN
1,8280,Craig Gordon,Scotland,6.412778,9,0,0,0,0.0,0.0,NaN,0.444444,52.843902,0.0,NaN,NaN,NaN,NaN,1.055556,NaN,NaN,NaN,NaN,0.333333,NaN,NaN,0.500000,27.666667,NaN,6.055556,NaN
2,11119,Lionel Messi,Argentina,7.812119,41,12,25,8,3.0,0.0,3.950000,0.171429,83.431002,0.0,0.946429,0.142857,0.435714,NaN,0.250000,1.059524,0.250000,2.516667,2.609524,1.604762,0.454762,2.223810,1.290476,42.352381,1.092857,1.976190,0.642857
3,13754,Manuel Neuer,Germany,6.393606,43,0,0,0,3.0,0.0,NaN,0.045455,79.393175,0.0,NaN,NaN,NaN,NaN,0.863636,0.090909,NaN,NaN,0.090909,0.393939,NaN,NaN,NaN,30.779545,NaN,6.434091,NaN
4,20874,Luka Modric,Croatia,6.715343,55,4,4,5,4.0,0.0,0.876634,0.411765,89.435256,0.0,0.736111,0.687908,0.421569,0.058824,0.698529,0.437092,0.238971,1.097222,0.437500,0.545752,NaN,0.301471,0.613562,46.213235,0.687092,2.975490,0.264706



--- DataFrame Info after column reorder ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1208 entries, 0 to 1207
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Player_ID                   1208 non-null   int64  
 1   Player_Name                 1208 non-null   object 
 2   Team_Name                   1157 non-null   object 
 3   rating                      924 non-null    float64
 4   apps                        1208 non-null   int64  
 5   manOfTheMatch               1208 non-null   int64  
 6   goal                        1208 non-null   int64  
 7   assistTotal                 1208 non-null   int64  
 8   yellowCard                  1208 non-null   float64
 9   redCard                     1208 non-null   float64
 10  shotsPerGame                814 non-null    float64
 11  aerialWonPerGame            879 non-null    float64
 12  passSuccess                 924 non-null    f

In [ ]:
merged_df.to_csv('merged_player_stats.csv', index=False)

from google.colab import files
files.download('merged_player_stats.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>